# Test Cases

In [1]:
!pip install -q llama-index-llms-anthropic
from pydantic import BaseModel, Field
from typing import List, Literal
from llama_index.llms.anthropic import Anthropic
from llama_index.core.prompts import PromptTemplate
import pandas as pd
from dotenv import load_dotenv
import os
from llama_index.core.program import FunctionCallingProgram

    yapf (>='0.28') ; python_version < "3.6"
         ~^


In [2]:
load_dotenv()

True

## Testcase Generation

### Response Model

In [3]:
class QAPair(BaseModel):
    question: str = Field(description="The generated question.")
    reference_answer: str = Field(
        description=(
            "A concise answer grounded directly in the document text. "
            "For unanswerable questions, must be exactly: "
            "'The document does not contain this information.'"
        )
    )

class QADataset(BaseModel):
    qa_pairs: List[QAPair] = Field(description="List of generated question-answer pairs.")

### Question Types

In [4]:
QUESTION_TYPES = [
    {
        "type": "single_hop_factual",
        "count": 2,
        "instruction": (
            "Generate {count} single-hop factual questions. Each question should have "
            "a clear, specific answer located in one place in the document. "
            "Example style: 'When was the university established?'"
        ),
    },
    {
        "type": "summarization",
        "count": 2,
        "instruction": (
            "Generate {count} summarization questions that require collecting information "
            "from a broader section of the document but still answerable via single-hop retrieval. "
            "Example style: 'What are the key admission requirements for the CS programme?'"
        ),
    },
    {
        "type": "unanswerable",
        "count": 2,
        "instruction": (
            "Generate {count} unanswerable questions — questions that sound plausible and "
            "related to the document's topic, but whose answers are genuinely NOT present "
            "in the document. The reference_answer for these MUST be exactly: "
            "'The document does not contain this information.'"
        ),
    },
    {
        "type": "needle_in_haystack",
        "count": 2,
        "instruction": (
            "Generate {count} needle-in-a-haystack questions targeting very specific details "
            "buried in the document — a single number, a specific name mentioned once, "
            "or a minor detail that is easy to miss."
        ),
    },
    {
        "type": "multi_task",
        "count": 2,
        "instruction": (
            "Generate {count} multi-task questions where a single question asks for "
            "two to four distinct pieces of information from the document. "
            "Example style:\n"
            "Please answer the following three questions:\n"
            "1. How many departments are there?\n"
            "2. What are their telephone numbers?\n"
            "3. Who is the current president?"
        )
    }
]

### Prompt Template

In [5]:
PROMPT_TEMPLATE = PromptTemplate(
    "You are a test dataset generator for evaluating RAG systems.\n"
    "Given a document, generate question-answer pairs of a specified type.\n\n"
    "Rules:\n"
    "- All questions must require only single-hop reasoning (except unanswerable ones).\n"
    "- Reference answers must be concise and directly grounded in the document text.\n"
    "- For unanswerable questions, the reference_answer must be exactly: "
    "'The document does not contain this information.'\n\n"
    "Document name: {doc_name}\n\n"
    "--- DOCUMENT START ---\n"
    "{doc_text}\n"
    "--- DOCUMENT END ---\n\n"
    "Task: {instruction}\n"
    "Generate exactly {count} question-answer pairs.\n"
)

### Generator

In [6]:
def load_document(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [7]:
def generate_questions(doc_name, doc_text, qt_config):
    instruction = qt_config["instruction"].format(count=qt_config["count"])

    program = FunctionCallingProgram.from_defaults(
        output_cls=QADataset,
        prompt=PROMPT_TEMPLATE,
        llm=llm,
    )

    result = program(
        doc_name=doc_name,
        doc_text=doc_text,
        instruction=instruction,
        count=qt_config["count"],
    )

    rows = []
    for pair in result.qa_pairs:
        rows.append({
            "document": doc_name,
            "question_type": qt_config["type"],
            "question": pair.question,
            "reference_answer": pair.reference_answer,
        })
    return rows

In [8]:
llm = Anthropic(
    model="claude-opus-4-6",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
    max_tokens=4096
)

doc_texts = {
    "Wiki": load_document("../dataset/City_University_of_Hong_Kong.md"),
    "Curriculum": load_document("../dataset/CS_Curriculum_Clean.md"),
    "Magazine": load_document("../dataset/City_Magazine_Clean.md"),
}

output_format = """
Output format:
[
  {
    "question": "...",
    "reference_answer": "...",
    "question_type": "..."
  }
]"""

# Generate all questions
all_questions = []

for doc_name, doc_text in doc_texts.items():
    for qt_config in QUESTION_TYPES:
        print(f"Generating {qt_config['type']} for {doc_name}...")
        rows = generate_questions(doc_name, doc_text, qt_config)
        all_questions.extend(rows)
        print(f"  → {len(rows)} questions generated")

# --- Output ---
df = pd.DataFrame(all_questions)
print(f"\nTotal: {len(df)} questions")
display(df.groupby(["document", "question_type"]).size().unstack(fill_value=0))

df.to_csv("test_dataset.csv", index=False)
df.to_json("test_dataset.json", orient="records", indent=2)

Generating single_hop_factual for Wiki...
  → 2 questions generated
Generating summarization for Wiki...
  → 2 questions generated
Generating unanswerable for Wiki...
  → 2 questions generated
Generating needle_in_haystack for Wiki...
  → 2 questions generated
Generating multi_task for Wiki...
  → 2 questions generated
Generating single_hop_factual for Curriculum...
  → 2 questions generated
Generating summarization for Curriculum...
  → 2 questions generated
Generating unanswerable for Curriculum...
  → 2 questions generated
Generating needle_in_haystack for Curriculum...
  → 2 questions generated
Generating multi_task for Curriculum...
  → 2 questions generated
Generating single_hop_factual for Magazine...
  → 2 questions generated
Generating summarization for Magazine...
  → 2 questions generated
Generating unanswerable for Magazine...
  → 2 questions generated
Generating needle_in_haystack for Magazine...
  → 2 questions generated
Generating multi_task for Magazine...
  → 2 questio

question_type,multi_task,needle_in_haystack,single_hop_factual,summarization,unanswerable
document,,,,,
Curriculum,2,2,2,2,2
Magazine,2,2,2,2,2
Wiki,2,2,2,2,2
